In [1]:
# pip install spacy
# python -m spacy download pt_core_news_sm
# pip install rdflib

In [2]:
# ======================================================================
# Bibliotecas Necessárias para o Projeto
# ======================================================================

import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import json
import pickle
import re
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter

In [3]:
DIR_OUTPUT = "C:\\Users\\alexa\\Atividades\\LocalDeepSeek\\Output\\"
DIR_DATASET = "C:\\Users\\alexa\\Atividades\\LocalDeepSeek\\Dataset\\"

In [4]:

# ======================================================================
# PARTE 1: PARSER TTL MANUAL (SEM RDFLIB)
# ======================================================================

class TTLParser:   
    def __init__(self):
        self.prefixes = {}
        self.triplas = []
        self.classes = {}
        self.propriedades = {}
        self.instancias = {}
        
    def parse_file(self, arquivo):
        # print(f"\n📂 Lendo arquivo: {arquivo}")
        
        arquivo = DIR_DATASET + arquivo
        with open(arquivo, 'r', encoding='utf-8') as f:
            linhas = f.readlines()
        
        # Fase 1: Extrair prefixos
        self._extrair_prefixes(linhas)
        
        # Fase 2: Extrair triplas
        self._extrair_triplas(linhas)
        
        # Fase 3: Organizar por tipo
        self._organizar_dados()
        
        return self
    
    def _extrair_prefixes(self, linhas):
        for linha in linhas:
            if linha.strip().startswith('@prefix'):
                match = re.match(r'@prefix\s+(\w*):\s+<([^>]+)>', linha)
                if match:
                    prefix, uri = match.groups()
                    self.prefixes[prefix] = uri
    
    def _limpar_valor(self, valor):
        # Remove language tags @pt, @en
        valor = re.sub(r'@\w+$', '', valor)

        # Remove type annotations ^^xsd:date etc
        valor = re.sub(r'\^\^[\w:]+$', '', valor)

        # Remove aspas
        valor = valor.strip('"')
        return valor
    
    def _extrair_triplas(self, linhas):
        buffer = ""        
        for linha in linhas:
            # Ignorar comentários e linhas vazias
            if linha.strip().startswith('#') or not linha.strip():
                continue
            
            # Ignorar declarações de prefix
            if linha.strip().startswith('@prefix'):
                continue
            
            buffer += " " + linha.strip()
            
            # Quando encontrar ponto final, processar buffer
            if buffer.strip().endswith('.'):
                self._processar_bloco(buffer)
                buffer = ""
    
    def _processar_bloco(self, bloco):
        bloco = bloco.strip().rstrip('.')
        
        # Dividir por ponto-e-vírgula (mesmo sujeito)
        if ';' in bloco:
            partes = bloco.split(None, 1)
            if len(partes) < 2:
                return
            
            sujeito = partes[0]
            resto = partes[1]
            
            # Dividir predicado-objeto por ;
            pares = re.split(r'\s*;\s*', resto)
            
            for par in pares:
                par = par.strip()
                if not par:
                    continue
                
                # Dividir predicado de objetos
                partes_par = par.split(None, 1)
                if len(partes_par) < 2:
                    continue
                
                predicado = partes_par[0]
                objetos_str = partes_par[1]
                
                # Múltiplos objetos separados por vírgula
                objetos = re.split(r'\s*,\s*', objetos_str)
                
                for objeto in objetos:
                    objeto = objeto.strip()
                    if objeto:
                        self.triplas.append((
                            self._limpar_termo(sujeito),
                            self._limpar_termo(predicado),
                            self._limpar_termo(objeto)
                        ))
        else:
            # Tripla simples: sujeito predicado objeto
            partes = bloco.split(None, 2)
            if len(partes) == 3:
                s, p, o = partes
                self.triplas.append((
                    self._limpar_termo(s),
                    self._limpar_termo(p),
                    self._limpar_termo(o)
                ))
    
    def _limpar_termo(self, termo):
        termo = termo.strip()
        
        # Remover aspas
        if termo.startswith('"'):
            # Valor literal
            return self._limpar_valor(termo)
        
        # Remover prefix completo, manter apenas nome local
        if ':' in termo and not termo.startswith('http'):
            return termo.split(':')[-1]
        
        return termo
    
    def _organizar_dados(self):
       
        # Primeiro passo: identificar classes
        for s, p, o in self.triplas:
            if p == 'a' and o == 'Class':
                self.classes[s] = {'tipo': 'Class', 'label': s}
            elif p == 'subClassOf':
                if s not in self.classes:
                    self.classes[s] = {'tipo': 'Class', 'label': s}
                self.classes[s]['subClassOf'] = o
        
        # Segundo passo: identificar propriedades
        for s, p, o in self.triplas:
            if p == 'a' and ('Property' in o):
                self.propriedades[s] = {'tipo': o, 'label': s}
        
        # Terceiro passo: identificar instâncias (não são classes nem propriedades)
        for s, p, o in self.triplas:
            if p == 'a' and o not in ['Class', 'Ontology'] and 'Property' not in o:
                if s not in self.classes and s not in self.propriedades:
                    self.instancias[s] = {'tipo': o, 'triplas': []}
        
        # Quarto passo: agrupar triplas por instância
        for s, p, o in self.triplas:
            if s in self.instancias:
                if 'triplas' not in self.instancias[s]:
                    self.instancias[s]['triplas'] = []
                self.instancias[s]['triplas'].append((p, o))

# ======================================================================
# PARTE 2: CONSTRUÇÃO DO GRAFO NETWORKX
# ======================================================================

def construir_grafo_de_ontologia(arquivo_ttl):
    
    # Parsear TTL
    parser = TTLParser()
    parser.parse_file(arquivo_ttl)
    
    # Criar grafo direcionado
    G = nx.DiGraph()
    
    # Adicionar instâncias como nós
    for inst_id, dados in parser.instancias.items():
        
        # Atributos do nó
        atributos = {
            'id': inst_id,
            'tipo': dados.get('tipo', 'Desconhecido')
        }
        
        # Processar triplas da instância
        for predicado, objeto in dados.get('triplas', []):
            
            # Se for label
            if predicado == 'label':
                atributos['label'] = objeto
            
            # Outros atributos literais
            elif objeto not in parser.instancias:
                # É um atributo literal (não é outra instância)
                atributos[predicado] = objeto
        
        # Adicionar nó
        if 'label' not in atributos:
            atributos['label'] = inst_id
        
        G.add_node(inst_id, **atributos)
    num_arestas = 0
    
    for inst_id, dados in parser.instancias.items():
        for predicado, objeto in dados.get('triplas', []):
            
            # Se objeto é outra instância, criar aresta
            if objeto in parser.instancias or objeto in G.nodes():
                
                # Não criar arestas para 'a' (tipo)
                if predicado not in ['a', 'label']:
                    G.add_edge(inst_id, objeto, relacao=predicado)
                    num_arestas += 1
    
    return G, parser

# ======================================================================
# PARTE 3: QUERIES ESPECIALIZADAS
# ======================================================================

def query_01_perfil_alan_turing(G):
    
    print("# ======================================================================")
    print("# QUERY 1: Perfil Completo de Alan Turing")
    print("# ======================================================================")
    
    node_id = 'AlanTuring'
    
    if node_id not in G:
        print("# Alan Turing não encontrado no grafo")
        return None
    
    # Dados do nó
    dados = dict(G.nodes[node_id])
    
    # Relações de saída
    print("# ======================================================================")    
    print(f"# RELAÇÕES (Total: {G.out_degree(node_id)}):")
    print("# ======================================================================")

    relacoes_por_tipo = defaultdict(list)
    for destino in G.successors(node_id):
        edge_data = G[node_id][destino]
        relacao = edge_data.get('relacao', 'relacao')
        label_destino = G.nodes[destino].get('label', destino)
        relacoes_por_tipo[relacao].append(label_destino)
    
    for relacao, destinos in sorted(relacoes_por_tipo.items()):
        print(f"\n  • {relacao}:")
        for destino in destinos:
            print(f"    - {destino}")
    
    resultado = {
        'node_id': node_id,
        'dados': dados,
        'grau': G.degree(node_id),
        'relacoes_saida': {k: v for k, v in relacoes_por_tipo.items()}
    }
    
    return resultado

def query_02_areas_conhecimento(G):
    print("# ======================================================================")    
    print("# QUERY 2: Áreas de Conhecimento")
    print("# ======================================================================")    
    
    areas = []
    
    for node in G.nodes():
        if G.nodes[node].get('tipo') == 'AreaConhecimento':
            areas.append({
                'id': node,
                'label': G.nodes[node].get('label', node),
                'nomeArea': G.nodes[node].get('nomeArea', 'N/A')
            })
    
    # print(f"\n📚 Total de áreas: {len(areas)}")
    for i, area in enumerate(areas, 1):
        print(f"  {i}. {area['label']}")
    
    return areas

def query_03_invencoes(G):   
    print("# ======================================================================")  
    print("# QUERY 3: Invenções de Alan Turing")
    print("# ======================================================================")  
    
    invencoes = []
    
    if 'AlanTuring' in G:
        for destino in G.successors('AlanTuring'):
            edge_data = G['AlanTuring'][destino]
            if edge_data.get('relacao') == 'inventou':
                invencoes.append({
                    'id': destino,
                    'label': G.nodes[destino].get('label', destino),
                    'tipo': G.nodes[destino].get('tipo', 'N/A'),
                    'ano': G.nodes[destino].get('anoConstrucao', 'N/A')
                })
    
    print(f"\n🔧 Total de invenções: {len(invencoes)}")
    for i, inv in enumerate(invencoes, 1):
        print(f"  {i}. {inv['label']} ({inv['tipo']}) - Ano: {inv['ano']}")
    
    return invencoes

def query_04_instituicoes(G):   
    print("# ======================================================================") 
    print("# QUERY 4: Instituições - Estudo e Trabalho")
    print("# ======================================================================") 
    
    instituicoes = {'estudou': [], 'trabalhou': []}
    
    if 'AlanTuring' in G:
        for destino in G.successors('AlanTuring'):
            edge_data = G['AlanTuring'][destino]
            relacao = edge_data.get('relacao', '')
            
            if relacao in ['estudouEm', 'doutoradoEm']:
                instituicoes['estudou'].append({
                    'id': destino,
                    'label': G.nodes[destino].get('label', destino),
                    'relacao': relacao
                })
            elif relacao == 'trabalhouEm':
                instituicoes['trabalhou'].append({
                    'id': destino,
                    'label': G.nodes[destino].get('label', destino)
                })
    
    print(f"\n🎓 ESTUDOU/DOUTORADO ({len(instituicoes['estudou'])}):")
    for inst in instituicoes['estudou']:
        print(f"  • {inst['label']} ({inst['relacao']})")
    
    print(f"\n💼 TRABALHOU ({len(instituicoes['trabalhou'])}):")
    for inst in instituicoes['trabalhou']:
        print(f"  • {inst['label']}")
    
    return instituicoes

def query_05_locais(G):   
    print("# ======================================================================") 
    print("# QUERY 5: Locais (Nascimento, Morte, Vida)")
    print("# ======================================================================") 
    
    locais = {}
    
    if 'AlanTuring' in G:
        for destino in G.successors('AlanTuring'):
            edge_data = G['AlanTuring'][destino]
            relacao = edge_data.get('relacao', '')
            
            if relacao in ['nasceuEm', 'morreuEm', 'viveuEm']:
                locais[relacao] = {
                    'id': destino,
                    'label': G.nodes[destino].get('label', destino),
                    'tipo': G.nodes[destino].get('tipo', 'N/A')
                }
    
    for relacao, dados in locais.items():
        print(f"\n  • {relacao}: {dados['label']} ({dados['tipo']})")
    
    return locais

def query_06_eventos(G):
   
    print("# ======================================================================")
    print("# QUERY 6: Eventos Históricos")
    print("# ======================================================================")
    
    eventos = []
    
    if 'AlanTuring' in G:
        for destino in G.successors('AlanTuring'):
            tipo_destino = G.nodes[destino].get('tipo', '')
            if 'Evento' in tipo_destino or destino == 'SegundaGuerraMundial':
                eventos.append({
                    'id': destino,
                    'label': G.nodes[destino].get('label', destino),
                    'dataInicio': G.nodes[destino].get('dataInicio', 'N/A'),
                    'dataFim': G.nodes[destino].get('dataFim', 'N/A')
                })
    
    # print(f"\n📅 Total de eventos: {len(eventos)}")
    for evento in eventos:
        print(f"  • {evento['label']}")
        print(f"    Período: {evento['dataInicio']} a {evento['dataFim']}")
    
    return eventos

def query_07_publicacoes(G):
   
    print("# ======================================================================")
    print("# QUERY 7: Publicações")
    print("# ======================================================================")
    
    publicacoes = []
    
    if 'AlanTuring' in G:
        for destino in G.successors('AlanTuring'):
            edge_data = G['AlanTuring'][destino]
            if edge_data.get('relacao') == 'publicou':
                publicacoes.append({
                    'id': destino,
                    'label': G.nodes[destino].get('label', destino),
                    'titulo': G.nodes[destino].get('titulo', 'N/A'),
                    'data': G.nodes[destino].get('dataPublicacao', 'N/A')
                })
    
    # print(f"\n📄 Total de publicações: {len(publicacoes)}")
    for pub in publicacoes:
        print(f"  • {pub['label']} ({pub['data']})")
    
    return publicacoes

def query_08_rede_completa(G):
    print("# ======================================================================")
    print("# QUERY 8: Rede Completa (Vizinhança)")
    print("# ======================================================================")
    
    if 'AlanTuring' not in G:
        return {}
    
    # Vizinhança de 1 nível
    vizinhos = set(G.successors('AlanTuring'))
    
    # Vizinhança de 2 níveis
    vizinhos_2 = set()
    for v in vizinhos:
        vizinhos_2.update(G.successors(v))
    
    # Tipos na rede
    tipos = defaultdict(int)
    for node in vizinhos | vizinhos_2:
        tipo = G.nodes[node].get('tipo', 'Desconhecido')
        tipos[tipo] += 1
    
    # print(f"\n📊 TIPOS NA REDE:")
    for tipo, count in sorted(tipos.items(), key=lambda x: x[1], reverse=True):
        print(f"  • {tipo}: {count}")
    
    return {
        'nivel_1': list(vizinhos),
        'nivel_2': list(vizinhos_2),
        'tipos': dict(tipos)
    }

def query_09_caminho_entre_nos(G, origem, destino):
   
    print("# ======================================================================")
    print(f"# QUERY 9: Caminho entre {origem} e {destino}")
    print("# ======================================================================")
    
    try:
        caminho = nx.shortest_path(G, origem, destino)
        for i, node in enumerate(caminho):
            label = G.nodes[node].get('label', node)
            print(f"  {i}. {label}")
            
            if i < len(caminho) - 1:
                # Mostrar relação
                next_node = caminho[i+1]
                if G.has_edge(node, next_node):
                    relacao = G[node][next_node].get('relacao', '→')
                    print(f"     └─ [{relacao}] ─→")
        
        return caminho
    
    except nx.NetworkXNoPath:
        print(f"#  Não há caminho entre {origem} e {destino}")
        return None

def query_10_estatisticas_globais(G):
   
    print("# ======================================================================")
    print("# QUERY 10: Estatísticas Globais")
    print("# ======================================================================")
   
    # Graus
    degrees = dict(G.degree())
    grau_medio = sum(degrees.values()) / len(degrees) if degrees else 0
    
    # Top nós
    top_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:5]
    for i, (node, degree) in enumerate(top_nodes, 1):
        label = G.nodes[node].get('label', node)
        print(f"  {i}. {label}: {degree} conexões")
    
    # Tipos
    tipos = defaultdict(int)
    for node in G.nodes():
        tipo = G.nodes[node].get('tipo', 'Desconhecido')
        tipos[tipo] += 1
    
    print(f"\n🏷️  TIPOS DE NÓS:")
    for tipo, count in sorted(tipos.items(), key=lambda x: x[1], reverse=True):
        print(f"  • {tipo}: {count}")
    
    # Relações
    relacoes = defaultdict(int)
    for u, v, data in G.edges(data=True):
        rel = data.get('relacao', 'sem_relacao')
        relacoes[rel] += 1
    
    print("# ======================================================================")
    print(f"# TIPOS DE RELAÇÕES (Top 10):")
    print("# ======================================================================")
    top_rel = sorted(relacoes.items(), key=lambda x: x[1], reverse=True)[:10]
    for rel, count in top_rel:
        print(f"  • {rel}: {count}")
    
    return {
        'num_nodes': G.number_of_nodes(),
        'num_edges': G.number_of_edges(),
        'density': nx.density(G),
        'grau_medio': grau_medio,
        'tipos': dict(tipos),
        'relacoes': dict(relacoes)
    }

# ======================================================================
# PARTE 4: VISUALIZAÇÃO
# ======================================================================

def visualizar_grafo_ontologia(G, titulo, arquivo, subgrafo_nodes=None):
   
    # Se for subgrafo
    if subgrafo_nodes:
        G_viz = G.subgraph(subgrafo_nodes).copy()
    else:
        G_viz = G
    
    if G_viz.number_of_nodes() == 0:
        print("# Grafo vazio!")
        return
    
    # Figura
    fig, ax = plt.subplots(figsize=(20, 14), dpi=150)
    ax.set_facecolor('#F8F9FA')
    fig.patch.set_facecolor('white')
    
    # Layout
    pos = nx.spring_layout(G_viz, k=3, iterations=50, seed=42)
    
    # Cores por tipo
    cores_tipo = {
        'Matematico': '#FF6B6B',
        'CientistaDaComputacao': '#4ECDC4',
        'Criptografo': '#96CEB4',
        'AreaConhecimento': '#45B7D1',
        'DispositivoCriptografico': '#FD79A8',
        'MaquinaComputacional': '#FFD93D',
        'Conceito': '#6BCB77',
        'Universidade': '#FFEAA7',
        'Cidade': '#DFE6E9',
        'Pais': '#B2BEC3',
        'Edificio': '#74B9FF',
        'EventoHistorico': '#55EFC4',
        'OrganizacaoMilitar': '#FDA7DF',
        'Artigo': '#C7CEEA',
        'Teoria': '#F8B500',
    }
    
    node_colors = [cores_tipo.get(G_viz.nodes[n].get('tipo', ''), '#95A5A6') 
                   for n in G_viz.nodes()]
    
    # Tamanhos
    node_sizes = [400 + (G_viz.degree(n) * 150) for n in G_viz.nodes()]
    
    # Desenhar
    nx.draw_networkx_edges(G_viz, pos, edge_color='#2C3E50', width=2, 
                          alpha=0.5, arrows=True, arrowsize=20,
                          arrowstyle='->', connectionstyle='arc3,rad=0.1', ax=ax)
    
    nx.draw_networkx_nodes(G_viz, pos, node_color=node_colors, 
                          node_size=node_sizes, alpha=0.95,
                          linewidths=2.5, edgecolors='white', ax=ax)
    
    # Labels
    labels = {n: G_viz.nodes[n].get('label', n)[:25] for n in G_viz.nodes()}
    nx.draw_networkx_labels(G_viz, pos, labels, font_size=9, 
                           font_weight='bold', font_color='black', ax=ax)
    
    # Título
    ax.set_title(titulo, fontsize=24, fontweight='bold', pad=25, color='#2C3E50')
    
    # Stats
    stats = f"Nós: {G_viz.number_of_nodes()} | Arestas: {G_viz.number_of_edges()}"
    ax.text(0.5, 0.02, stats, transform=ax.transAxes, 
           ha='center', fontsize=12, fontweight='bold',
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    ax.axis('off')
    plt.tight_layout()
    
    plt.savefig(DIR_OUTPUT + arquivo, dpi=300, bbox_inches='tight')
    
    plt.show()
    plt.close()

# ======================================================================
# PARTE 5: SALVAR RESULTADOS
# ======================================================================

def salvar_resultados_queries(resultados, arquivo='resultados_queries.json'):
    
    # Converter para formato serializável
    resultados_serializaveis = {}
    
    for query_name, resultado in resultados.items():
        if isinstance(resultado, dict):
            resultados_serializaveis[query_name] = resultado
        elif isinstance(resultado, list):
            resultados_serializaveis[query_name] = resultado
        else:
            resultados_serializaveis[query_name] = str(resultado)
    
    arquivo = DIR_OUTPUT + arquivo
    with open(arquivo, 'w', encoding='utf-8') as f:
        json.dump(resultados_serializaveis, f, indent=2, ensure_ascii=False)

def salvar_grafo(G, arquivo_base):
    
    # Pickle
    arquivo_base = DIR_OUTPUT + arquivo_base
    with open(f"{arquivo_base}.pkl", 'wb') as f:
        pickle.dump(G, f)
    # print(f"  ✓ Pickle: {arquivo_base}.pkl")
    
    # JSON
    from networkx.readwrite import json_graph
    data = json_graph.node_link_data(G, edges='edges')

    with open(f"{arquivo_base}.json", 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    # print(f"  ✓ JSON: {arquivo_base}.json")

def gerar_relatorio_completo(G, resultados, arquivo='relatorio_ontologia.txt'):
    
    arquivo = DIR_OUTPUT + arquivo
    with open(arquivo, 'w', encoding='utf-8') as f:
        f.write("="*80 + "\n")
        f.write("RELATÓRIO COMPLETO - GRAFO DE CONHECIMENTO\n")
        f.write("Ontologia: Alan Turing\n")
        f.write(f"Data: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*80 + "\n\n")
        
        # Estatísticas
        f.write("ESTATÍSTICAS DO GRAFO:\n")
        f.write(f"  • Nós: {G.number_of_nodes()}\n")
        f.write(f"  • Arestas: {G.number_of_edges()}\n")
        f.write(f"  • Densidade: {nx.density(G):.4f}\n\n")
        
        # Resultados das queries
        f.write("RESULTADOS DAS QUERIES:\n")
        f.write("-"*80 + "\n\n")
        
        for query_name, resultado in resultados.items():
            f.write(f"{query_name}:\n")
            f.write(f"  {resultado}\n\n")

# ======================================================================
# FUNÇÃO PRINCIPAL
# ======================================================================

def main():
   
    # Arquivo TTL
    arquivo_ttl = "ontologia_otimizada.ttl"
    
    # Construir grafo
    G, parser = construir_grafo_de_ontologia(arquivo_ttl)
    
    # Executar queries
    print("# ======================================================================")
    print("# EXECUTANDO QUERIES")
    print("# ======================================================================")
    
    resultados = {}
    
    # Query 1
    resultados['query_01_perfil_alan_turing'] = query_01_perfil_alan_turing(G)
    
    # Query 2
    resultados['query_02_areas_conhecimento'] = query_02_areas_conhecimento(G)
    
    # Query 3
    resultados['query_03_invencoes'] = query_03_invencoes(G)
    
    # Query 4
    resultados['query_04_instituicoes'] = query_04_instituicoes(G)
    
    # Query 5
    resultados['query_05_locais'] = query_05_locais(G)
    
    # Query 6
    resultados['query_06_eventos'] = query_06_eventos(G)
    
    # Query 7
    resultados['query_07_publicacoes'] = query_07_publicacoes(G)
    
    # Query 8
    resultados['query_08_rede_completa'] = query_08_rede_completa(G)
    
    # Query 9
    if 'AlanTuring' in G and 'CienciaComputacao' in G:
        resultados['query_09_caminho'] = query_09_caminho_entre_nos(
            G, 'AlanTuring', 'CienciaComputacao'
        )
    
    # Query 10
    resultados['query_10_estatisticas'] = query_10_estatisticas_globais(G)
    
    # # Visualização 1: Grafo completo
    # visualizar_grafo_ontologia(
    #     G,
    #     "Grafo Completo - Ontologia Alan Turing",
    #     "grafo_ontologia_completo.png"
    # )
    
    # # Visualização 2: Subgrafo de Alan Turing
    # if 'AlanTuring' in G:
    #     vizinhos = set(G.successors('AlanTuring'))
    #     vizinhos.add('AlanTuring')
    #     visualizar_grafo_ontologia(
    #         G,
    #         "Rede de Alan Turing (Vizinhança Direta)",
    #         "grafo_ontologia_turing.png",
    #         subgrafo_nodes=vizinhos
    #     )
    
    # salvar_resultados_queries(resultados)
    # salvar_grafo(G, 'grafo_ontologia')
    # gerar_relatorio_completo(G, resultados)

if __name__ == "__main__":
    main()

# ======================================================================
# EXECUTANDO QUERIES
# ======================================================================
# ======================================================================
# QUERY 1: Perfil Completo de Alan Turing
# ======================================================================
# ======================================================================
# RELAÇÕES (Total: 15):
# ======================================================================

  • contribuiuPara:
    - Inteligência Artificial

  • desenvolveu:
    - Teoria da Computabilidade

  • doutoradoEm:
    - Universidade de Princeton

  • especializadoEm:
    - Ciência da Computação
    - Criptografia

  • estudouEm:
    - Universidade de Cambridge

  • inventou:
    - Bombe
    - Máquina de Turing

  • morreuEm:
    - Reino Unido

  • nasceuEm:
    - Londres

  • participouDe:
    - Segunda Guerra Mundial

  • propôs:
    - Teste de Turing

  • public

In [5]:
# ====================================================================== 
# FIM DO PROGRAMA
# ====================================================================== 